# 00 · Generate sample source data (optional)

Creates a small, deliberately imperfect retail dataset in the attached Lakehouse so the rest of the
pipeline (`01` → `04`) can be exercised end-to-end before pointing it at real client data.

It writes six Delta tables into the schema named in `TARGET_SCHEMA` (default `bronze`):

| Table | Rows | Notes |
|---|---|---|
| `regions` | 5 | Snowflaked parent of `stores` |
| `stores` | 40 | References `regions` |
| `customers` | 2,000 | ~3% null emails, a few duplicate natural keys |
| `products` | 300 | Category / subcategory hierarchy |
| `orders` | 20,000 | Header: customer, store, order + ship dates |
| `order_items` | ~60,000 | Line grain: quantity, unit price, discount; ~1% orphan product ids |

**Skip this notebook** when profiling real data — it exists only so you can see what the pipeline produces.

In [ ]:
# PARAMETERS — override from a pipeline or notebookutils.notebook.run()
TARGET_SCHEMA = "bronze"    # lakehouse schema to write into ("" = lakehouse root Tables/ for non-schema lakehouses)
LAKEHOUSE_ROOT = ""         # abfss://<workspace-id>@onelake.dfs.fabric.microsoft.com/<lakehouse-id> ; "" = default lakehouse
SEED = 42
N_CUSTOMERS = 2000
N_ORDERS = 20000

In [ ]:
import random, datetime as dt
from pyspark.sql import functions as F, types as T


def _lakehouse_root():
    if LAKEHOUSE_ROOT:
        return LAKEHOUSE_ROOT.rstrip("/")
    ctx = notebookutils.runtime.context
    ws, lh = ctx.get("defaultLakehouseWorkspaceId"), ctx.get("defaultLakehouseId")
    if not lh:
        raise RuntimeError("Attach a default Lakehouse to this notebook or set LAKEHOUSE_ROOT.")
    return f"abfss://{ws}@onelake.dfs.fabric.microsoft.com/{lh}"


ROOT = _lakehouse_root()
TABLES_DIR = f"{ROOT}/Tables" + (f"/{TARGET_SCHEMA}" if TARGET_SCHEMA else "")
print("Writing sample tables to", TABLES_DIR)
random.seed(SEED)

In [ ]:
regions = [(1, "Northeast", "US"), (2, "Southeast", "US"), (3, "Midwest", "US"), (4, "West", "US"), (5, "Ontario", "CA")]
stores = [(i, f"Store {i:03d}", random.choice(["Mall", "Street", "Outlet"]), random.randint(1, 5),
           dt.date(2015 + random.randint(0, 8), random.randint(1, 12), random.randint(1, 28))) for i in range(1, 41)]
first = ["Ava", "Liam", "Noah", "Mia", "Zoe", "Ethan", "Ivy", "Owen", "Leo", "Nora"]
last = ["Patel", "Garcia", "Kim", "Nguyen", "Okafor", "Silva", "Rossi", "Chen", "Meyer", "Haddad"]
customers = []
for i in range(1, N_CUSTOMERS + 1):
    fn, ln = random.choice(first), random.choice(last)
    email = None if random.random() < 0.03 else f"{fn}.{ln}{i}@example.com".lower()
    customers.append((i, fn, ln, email, random.choice(["Consumer", "Corporate", "Home Office"]),
                      random.choice(["Bronze", "Silver", "Gold", "Platinum"]),
                      dt.date(2016 + random.randint(0, 8), random.randint(1, 12), random.randint(1, 28)),
                      random.random() < 0.85))
customers += customers[:15]  # deliberate duplicate natural keys

cats = {"Apparel": ["Shirts", "Jackets", "Shoes"], "Electronics": ["Audio", "Wearables", "Accessories"],
        "Home": ["Kitchen", "Decor", "Bedding"]}
products = []
for i in range(1, 301):
    c = random.choice(list(cats)); s = random.choice(cats[c])
    cost = round(random.uniform(3, 200), 2)
    products.append((i, f"SKU-{i:05d}", f"{s} item {i}", c, s, cost, round(cost * random.uniform(1.3, 2.4), 2),
                     random.choice(["Active", "Active", "Active", "Discontinued"])))

orders, items = [], []
item_id = 1
for o in range(1, N_ORDERS + 1):
    od = dt.date(2023, 1, 1) + dt.timedelta(days=random.randint(0, 729))
    ship = None if random.random() < 0.04 else od + dt.timedelta(days=random.randint(1, 9))
    orders.append((o, f"SO-{o:07d}", random.randint(1, N_CUSTOMERS), random.randint(1, 40), od, ship,
                   random.choice(["Web", "Store", "Phone"]), random.choice(["Completed", "Completed", "Returned", "Cancelled"])))
    for _ in range(random.randint(1, 5)):
        pid = random.randint(1, 300) if random.random() > 0.01 else random.randint(301, 320)  # ~1% orphan product ids
        qty = random.randint(1, 6)
        price = round(random.uniform(5, 400), 2)
        disc = round(price * qty * random.choice([0, 0, 0, 0.05, 0.1, 0.2]), 2)
        items.append((item_id, o, pid, qty, price, disc, round(price * qty - disc, 2)))
        item_id += 1

frames = {
    "regions": spark.createDataFrame(regions, "region_id int, region_name string, country_code string"),
    "stores": spark.createDataFrame(stores, "store_id int, store_name string, store_format string, region_id int, opened_date date"),
    "customers": spark.createDataFrame(customers, "customer_id int, first_name string, last_name string, email string, segment string, loyalty_tier string, signup_date date, is_active boolean"),
    "products": spark.createDataFrame(products, "product_id int, sku string, product_name string, category string, subcategory string, unit_cost double, list_price double, status string"),
    "orders": spark.createDataFrame(orders, "order_id int, order_number string, customer_id int, store_id int, order_date date, ship_date date, channel string, order_status string"),
    "order_items": spark.createDataFrame(items, "order_item_id int, order_id int, product_id int, quantity int, unit_price double, discount_amount double, line_total double"),
}
for name, df in frames.items():
    df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(f"{TABLES_DIR}/{name}")
    print(f"  wrote {name:12s} {df.count():>7,} rows")
print("Done. Refresh the Lakehouse explorer to see the tables under", TARGET_SCHEMA or "Tables")